<h1 style="background: linear-gradient(to right, #395761, #7799B6); color: white; padding: 20px; border-radius: 10px; text-align: center; font-family: Arial, sans-serif; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
  Upload a multi-gigabyte fine-tuning dataset in parts
</h1>

Crusoe removed the per-upload size ceiling for fine-tuning data with a multipart upload path in the OpenAI-compatible Uploads API. This notebook is the companion to the announcement post, *Making Room for Bigger Fine-Tuning Datasets*. It takes a public chat dataset of about 1.2 GB, uploads it to Crusoe in 128 MiB parts in parallel, watches the parts land, completes the upload, and polls until Crusoe has assembled the parts into a file you can hand to a fine-tuning job.

The whole flow uses the `openai` Python package plus plain REST calls for the visibility endpoints. Nothing runs on a GPU here; the only account you need is a Crusoe API key.

<div align="center"><img src="assets/multipart-flow.png" alt="A training file split into parts on your machine, sent in parallel into an upload session on Crusoe, completed with ordered part ids and a checksum, and assembled into one file id" width="900"></div>

| <div align="center">Limit or lifetime</div> | <div align="center">Crusoe</div> |
| --- | --- |
| Max size per upload | 16 GiB |
| Max part size | 128 MiB |
| Upload session lifetime | 2 hours |
| Part lifetime | Tied to the session; cleaned up on completion, cancel, or expiry |
| Resulting file expiry | Persists by default; optional `expires_after` from 1 hour to 30 days |

API reference: [Uploads](https://docs.crusoecloud.com/api/managed-ai/#tag/Uploads) and [Fine-tuning](https://docs.crusoecloud.com/api/managed-ai/#tag/Fine-tuning) in the Managed AI API docs.

<h1 style="background: linear-gradient(to right, #395761, #7799B6); color: white; padding: 20px; border-radius: 10px; text-align: center; font-family: Arial, sans-serif; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
  Step 0: Setup
</h1>

### Get a Crusoe API key

1. From the [Crusoe Console](https://console.crusoecloud.com/), select the organization name in the top left corner and open **Manage Organization**.
2. Under **Intelligence Foundry**, open **Security**, then [Inference API keys](https://console.crusoecloud.com/security/inference-api-keys).
3. Create a key, copy it, and paste it into `CRUSOE_API_KEY` in the next cell.

### Set up the environment

```bash
cd examples/training/multipart-dataset-upload
python3.12 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
```

The Uploads API is part of the OpenAI SDK from version 1.37 onward, so the client below is the same one you would use for files, fine-tuning jobs, and inference. `HF_TOKEN` is optional: the dataset used below is public, so leave it as `None` unless you swap in a gated dataset.

In [1]:
import hashlib
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests
from openai import OpenAI

CRUSOE_API_KEY = os.environ.get("CRUSOE_API_KEY", "paste-your-crusoe-api-key-here")
HF_TOKEN = None  # optional Hugging Face token, only for gated datasets

if "paste-your" in CRUSOE_API_KEY:
    raise ValueError("Add your Crusoe API key to CRUSOE_API_KEY above")

BASE_URL = "https://api.intelligence.crusoecloud.com/v1"
HEADERS = {"Authorization": f"Bearer {CRUSOE_API_KEY}"}

client = OpenAI(api_key=CRUSOE_API_KEY, base_url=BASE_URL)

MiB = 1024 * 1024
PART_SIZE = 128 * MiB  # Crusoe's ceiling per part
WORKERS = 8            # parts uploading at the same time; RAM use is about WORKERS * PART_SIZE

### Picking a part size and worker count

Part size is about control, not raw speed: a dropped connection costs one part, not the whole file, and every part that landed stays landed. On a stable, fast link use the biggest parts (128 MiB) for fewer requests. On a slow or flaky link use smaller parts so each retry is cheap. Part size does not change assembly time.

Parallel workers do change transfer time. The blog post's benchmark client moved a 3 GiB file in 23 parts about five times faster with 16 workers than with one. Each worker holds one part in memory, so `WORKERS * PART_SIZE` is the notebook's peak RAM for the upload.

<h1 style="background: linear-gradient(to right, #395761, #7799B6); color: white; padding: 20px; border-radius: 10px; text-align: center; font-family: Arial, sans-serif; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
  Step 1: Build a large training file
</h1>

The fine-tuning service expects chat-format JSONL: one JSON object per line with a `messages` list of `role` and `content` pairs. [HuggingFaceH4/ultrachat_200k](https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k) already stores conversations in that shape, and its `train_sft` split has about 208,000 of them, so writing the file is a matter of keeping the `messages` column.

The result is about 1.2 GB, ten 128 MiB parts, which is far past the size a single `client.files.create` call is meant for. Set `MAX_ROWS` to an integer to build a smaller file while you try the flow.

In [2]:
from datasets import load_dataset

DATASET = "HuggingFaceH4/ultrachat_200k"
SPLIT = "train_sft"
MAX_ROWS = None  # for example 20_000 rows for a quick run of about 115 MB

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
TRAIN_PATH = DATA_DIR / "train.jsonl"

dataset = load_dataset(DATASET, split=SPLIT, token=HF_TOKEN)
if MAX_ROWS:
    dataset = dataset.select(range(MAX_ROWS))

with TRAIN_PATH.open("w", encoding="utf-8") as f:
    for row in dataset:
        f.write(json.dumps({"messages": row["messages"]}, ensure_ascii=False) + "\n")

FILE_BYTES = TRAIN_PATH.stat().st_size
N_PARTS = -(-FILE_BYTES // PART_SIZE)  # ceiling division
print(f"{len(dataset):,} conversations")
print(f"{FILE_BYTES / MiB:,.0f} MiB on disk, {N_PARTS} parts of up to {PART_SIZE // MiB} MiB")

207,865 conversations
1,185 MiB on disk, 10 parts of up to 128 MiB


In [3]:
with TRAIN_PATH.open(encoding="utf-8") as f:
    first = json.loads(f.readline())

for message in first["messages"][:2]:
    print(f"{message['role']:>9}: {message['content'][:240]}")

     user: These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?
On your Collections pages & Featured Collections sections, you can easily show the secon
assistant: This feature only applies to Collection pages and Featured Collections sections of the section-based themes listed in the text material.


<h1 style="background: linear-gradient(to right, #395761, #7799B6); color: white; padding: 20px; border-radius: 10px; text-align: center; font-family: Arial, sans-serif; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
  Step 2: Create the upload
</h1>

An upload is a session that declares the file's name, purpose, size, and MIME type up front. The size matters: at completion, the bytes received across all parts must add up to exactly what you declared here. The session stays open for two hours, which is the window in which all parts and the completion call must happen.

In [4]:
upload = client.uploads.create(
    purpose="fine-tune",
    filename=TRAIN_PATH.name,
    bytes=FILE_BYTES,
    mime_type="text/jsonl",
)

print(f"Upload id : {upload.id}")
print(f"Status    : {upload.status}")
print(f"Declared  : {upload.bytes:,} bytes")
print(f"Session   : {(upload.expires_at - upload.created_at) // 3600} hours")

Upload id : upload_2916ee8e23d0481e97307d56c54bf95e
Status    : pending
Declared  : 1,242,536,347 bytes
Session   : 2 hours


<h1 style="background: linear-gradient(to right, #395761, #7799B6); color: white; padding: 20px; border-radius: 10px; text-align: center; font-family: Arial, sans-serif; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
  Step 3: Add the parts in parallel
</h1>

Each part is an independent request that returns a part id. Parts can go in any order and at the same time; you tell Crusoe the intended order when you complete the upload. The helper below reads one part from disk by offset, sends it, and re-sends only that part if the request fails. Every retry mints a fresh part id, and the abandoned one is cleaned up with the session.

The checksum is optional but cheap. Compute the MD5 of the whole file once and pass it at completion, and Crusoe verifies the assembled file against it.

In [5]:
def read_part(index):
    with TRAIN_PATH.open("rb") as f:
        f.seek(index * PART_SIZE)
        return f.read(PART_SIZE)


def send_part(index, attempts=3):
    data = read_part(index)
    for attempt in range(1, attempts + 1):
        try:
            part = client.uploads.parts.create(upload_id=upload.id, data=data)
            return index, part.id
        except Exception as exc:
            if attempt == attempts:
                raise
            print(f"part {index + 1} attempt {attempt} failed ({type(exc).__name__}), re-sending")


def md5_of(path):
    digest = hashlib.md5()
    with path.open("rb") as f:
        while chunk := f.read(8 * MiB):
            digest.update(chunk)
    return digest.hexdigest()


FILE_MD5 = md5_of(TRAIN_PATH)
print(f"MD5 {FILE_MD5}")

MD5 c331df73b21e3752cfea8758b11f26f4


In [6]:
part_ids = [None] * N_PARTS
started = time.time()

with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = [pool.submit(send_part, i) for i in range(N_PARTS)]
    for future in as_completed(futures):
        index, part_id = future.result()
        part_ids[index] = part_id
        print(f"[{time.time() - started:6.1f}s] part {index + 1:>2}/{N_PARTS} landed  {part_id}")

elapsed = time.time() - started
print(f"\n{FILE_BYTES / MiB:,.0f} MiB in {elapsed:.0f} s with {WORKERS} workers, about {FILE_BYTES / MiB / elapsed:.0f} MiB/s")

[  29.6s] part  1/10 landed  part_238ad6b25fd64aefb89ed1abc9e66c1b
[  29.6s] part  4/10 landed  part_ac56956761594d49881dcf798875078b
[  29.6s] part  6/10 landed  part_70d3763bbca1450db92c11b070b4fd2f
[  29.6s] part  2/10 landed  part_8e13015026d84f79845d792751cf1165


[  29.8s] part  8/10 landed  part_55e948cff2ab4bbcb59accedc259b711
[  29.8s] part  3/10 landed  part_f52c2bcb5be34c2fad6d892a2b23d113
[  30.0s] part  7/10 landed  part_fac5b6e6364e40a6a1b474fddd7277fe


[  30.2s] part  5/10 landed  part_8f0a9a7433ea4a4780f2c163de3c73fe


[  34.1s] part 10/10 landed  part_e791d63d71e841ad95432d83c876e618


[  37.6s] part  9/10 landed  part_eb672ec716b44f1285062d8d8c32e375

1,185 MiB in 38 s with 8 workers, about 32 MiB/s


### Check in on an upload in flight

Two visibility endpoints answer the questions a long upload raises. They are plain REST with the same API key. The parts list shows what Crusoe has received for this session, and the uploads list shows which sessions are still open. Run these from another process while a big upload is running, or after a kernel restart, to see where things stand.

In [7]:
received = requests.get(f"{BASE_URL}/uploads/{upload.id}/parts", headers=HEADERS).json()
print(f"{len(received['data'])} of {N_PARTS} parts received for {upload.id}")

pending = requests.get(f"{BASE_URL}/uploads", headers=HEADERS, params={"status": "pending"}).json()
print(f"\n{len(pending['data'])} pending upload session(s):")
for session in pending["data"]:
    print(f"  {session['id']}  {session['filename']}  {session['bytes']:,} bytes")

10 of 10 parts received for upload_2916ee8e23d0481e97307d56c54bf95e



1 pending upload session(s):
  upload_2916ee8e23d0481e97307d56c54bf95e  train.jsonl  1,242,536,347 bytes


<h1 style="background: linear-gradient(to right, #395761, #7799B6); color: white; padding: 20px; border-radius: 10px; text-align: center; font-family: Arial, sans-serif; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
  Step 4: Complete the upload
</h1>

Completion takes the part ids in file order plus the optional checksum. This is where Crusoe's behaviour diverges from OpenAI's: instead of blocking until the file exists, the call returns right away with status `pending`, and assembly runs in the background. That keeps your request from waiting on a multi-gigabyte merge, and it means an assembly in flight survives a deploy or a restart on Crusoe's side.

In [8]:
completion = client.uploads.complete(
    upload_id=upload.id,
    part_ids=part_ids,
    md5=FILE_MD5,
)
print(f"Status after complete: {completion.status}")

Status after complete: pending


<h1 style="background: linear-gradient(to right, #395761, #7799B6); color: white; padding: 20px; border-radius: 10px; text-align: center; font-family: Arial, sans-serif; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
  Step 5: Poll until the file is ready
</h1>

Poll the same retrieve endpoint until the status leaves `pending`. On success the `file` field carries an ordinary File object with the id you pass to a fine-tuning job. If assembly cannot finish, the status is `failed` and `error` says why, with a machine-readable code such as `size_mismatch`, `checksum_mismatch`, `missing_part`, or `invalid_jsonl`. The other terminal statuses are `cancelled`, after an explicit cancel, and `expired`, when a session passes two hours without completion.

In [9]:
def wait_for_assembly(upload_id, poll_seconds=5, deadline_seconds=1800):
    started = time.time()
    while time.time() - started < deadline_seconds:
        state = requests.get(f"{BASE_URL}/uploads/{upload_id}", headers=HEADERS).json()
        if state["status"] != "pending":
            return state, time.time() - started
        time.sleep(poll_seconds)
    raise TimeoutError(f"upload {upload_id} still pending after {deadline_seconds} s")


state, took = wait_for_assembly(upload.id)
if state["status"] != "completed":
    raise RuntimeError(f"assembly {state['status']}: {state.get('error')}")

TRAINING_FILE_ID = state["file"]["id"]
print(f"Assembled in about {took:.0f} s")
print(f"File id : {TRAINING_FILE_ID}")
print(f"Bytes   : {state['file']['bytes']:,}")

Assembled in about 17 s
File id : files:file_2916ee8e23d0481e97307d56c54bf95e:b2b31910-9093-4b1b-8aa4-2fce722b9898:9d4766182c8a052c43ab819b4f43097115be68ab
Bytes   : 1,242,536,347


The result is a regular file. It shows up in the same `files` endpoint as anything uploaded with a single `client.files.create` call. The parts do not outlive the session: Crusoe releases them shortly after completion, so the parts list drains to empty within a minute or so.

In [10]:
training_file = client.files.retrieve(TRAINING_FILE_ID)
print(f"{training_file.filename}  {training_file.bytes:,} bytes  purpose={training_file.purpose}  status={training_file.status}")

leftover = requests.get(f"{BASE_URL}/uploads/{upload.id}/parts", headers=HEADERS).json()
print(f"Parts still held for the session: {len(leftover['data'])} (released shortly after completion)")

train.jsonl  1,242,536,347 bytes  purpose=fine-tune  status=uploaded


Parts still held for the session: 6 (released shortly after completion)


<h1 style="background: linear-gradient(to right, #395761, #7799B6); color: white; padding: 20px; border-radius: 10px; text-align: center; font-family: Arial, sans-serif; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
  Step 6: Use the file in a fine-tuning job
</h1>

From here the file id goes wherever a file id goes today. The cell below shows the call that starts a LoRA fine-tune on the uploaded dataset. It is off by default because a job over 208,000 conversations is a real training run billed per training token; flip the flag when you mean it. The base model is resolved from the registry rather than hardcoded, since model ids are case sensitive and change with revisions.

In [11]:
SUBMIT_FINE_TUNING_JOB = False  # a full run over this dataset incurs real training charges
BASE_MODEL_NAME = "Qwen/Qwen3-8B"

if SUBMIT_FINE_TUNING_JOB:
    fine_tunable = [m for m in client.models.list().data if getattr(m, "fine_tuning_available", False)]
    base_model_id = next(m.id for m in fine_tunable if m.model_name == BASE_MODEL_NAME)
    job = client.fine_tuning.jobs.create(
        model=base_model_id,
        training_file=TRAINING_FILE_ID,
        suffix="ultrachat-multipart",
    )
    print(f"Job {job.id} is {job.status}")
    print(f"https://console.crusoecloud.com/foundry/fine-tuning/jobs/{job.id}/overview")
else:
    print("Skipped. Set SUBMIT_FINE_TUNING_JOB = True to train on the uploaded file.")

Skipped. Set SUBMIT_FINE_TUNING_JOB = True to train on the uploaded file.


<h1 style="background: linear-gradient(to right, #395761, #7799B6); color: white; padding: 20px; border-radius: 10px; text-align: center; font-family: Arial, sans-serif; text-shadow: 2px 2px 4px rgba(0,0,0,0.5);">
  Step 7: Cancel and clean up
</h1>

Sessions are self-cleaning. Cancel an upload you no longer want and its parts are released right away; anything you simply abandon expires with the two hour session and cleans itself up. The cell below opens a session and cancels it to show the shape of the call.

In [12]:
abandoned = client.uploads.create(
    purpose="fine-tune",
    filename="abandoned.jsonl",
    bytes=FILE_BYTES,
    mime_type="text/jsonl",
)
cancelled = client.uploads.cancel(abandoned.id)
print(f"{cancelled.id} is {cancelled.status}")

upload_cd57ec7379744a31827bed670ae78bc0 is cancelled


The assembled training file persists until you delete it. Keep it if you plan to run fine-tuning jobs against it; delete it when you are done with the walkthrough. The local `data/train.jsonl` and the Hugging Face download cache stay on disk until you remove them, and your API key stays in this notebook, so clear it before sharing the file.

In [13]:
DELETE_UPLOADED_FILE = False  # set to True to remove the assembled file from Crusoe

if DELETE_UPLOADED_FILE:
    deleted = client.files.delete(TRAINING_FILE_ID)
    print(f"Deleted {deleted.id}: {deleted.deleted}")
else:
    print("Skipped. Set DELETE_UPLOADED_FILE = True to delete the uploaded file.")

Skipped. Set DELETE_UPLOADED_FILE = True to delete the uploaded file.
